# 01 - Ingestion: parsing and chunking the Kazakh Labor Code + Tax Code

Corpus: two real laws from `adilet.zan.kz` (Russian legal text):

- **Labor Code of the Republic of Kazakhstan** (Law No. 414-V, 23 Nov 2015) - full text, 221 articles.
- **Tax Code of the Republic of Kazakhstan** (Law No. 214-VIII, effective 1 Jan 2026) - curated to the chapters a founding ML engineer at a fintech would actually be asked about: general provisions, Corporate Income Tax, Individual Income Tax, VAT, and special tax regimes for small business. 405 articles.

Both are legally public government legislation (freely republishable). Raw HTML snapshots live in `data/raw/`.

This notebook: load -> clean -> chunk (2 strategies) -> persist. The reusable parsing/chunking code lives in `scripts/ingest_lib.py` so `scripts/build_index.py` can run the identical pipeline non-interactively.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import ingest_lib as lib

labor_html = ROOT / "data/raw/labor_code_raw.html"
tax_html = ROOT / "data/raw/tax_code_raw.html"
print(labor_html.exists(), tax_html.exists())

True True


## 1. Load + clean

`ingest_lib` parses the adilet.zan.kz HTML structurally: article titles are `<p><b>Статья N. Title</b></p>` tags, bodies are sibling `<p>` tags, and chapter/section headings are `<h3>` tags. Walking the tag tree (instead of regex over flattened text) gives clean titles for free and avoids the header/footer boilerplate (site nav, ads, "how to cite this page") that a naive text dump would drag in - that boilerplate is simply outside the `<h3>`/`<p>` tags we walk.

In [2]:
labor_articles = lib.parse_labor_code(labor_html)
tax_articles = lib.parse_tax_code(tax_html)

print(f"Labor Code articles: {len(labor_articles)}")
print(f"Tax Code articles (curated chapters): {len(tax_articles)}")

lengths = [len(a.text) for a in labor_articles + tax_articles]
print(f"Article length chars: min={min(lengths)} max={max(lengths)} mean={sum(lengths)//len(lengths)}")
print(f"Total corpus size: {sum(lengths):,} chars (~{sum(lengths)//2000} pages at 2000 chars/page)")

Labor Code articles: 221
Tax Code articles (curated chapters): 405
Article length chars: min=112 max=47819 mean=2584
Total corpus size: 1,617,985 chars (~808 pages at 2000 chars/page)


In [3]:
sample = [a for a in labor_articles if a.article_number == "104"][0]
print("chapter:", sample.chapter)
print("title:  ", sample.title)
print("source: ", sample.source_file, "| doc version:", sample.document_version)
print("text:   ", sample.text[:300], "...")

chapter: Глава 8. НОРМИРОВАНИЕ И ОПЛАТА ТРУДА
title:   Установление минимального размера заработной платы
source:  labor_code_raw.html | doc version: adilet.zan.kz snapshot 2026-07-08
text:    Статья 104. Установление минимального размера заработной платы 1. Минимальный размер месячной заработной платы рассчитывается на основании методики определения минимального размера месячной заработной платы. Проект методики определения минимального размера месячной заработной платы подлежит рассмотр ...


## 2. Chunking - two independent strategies

- **Strategy A - `recursive_char`**: `RecursiveCharacterTextSplitter` (required by the RAID spec), applied *per article* so a chunk never straddles an article boundary (metadata stays exact). Character-based, no domain knowledge.
- **Strategy B - `article_based`**: one chunk per article, since Kazakh legal text is already organized into short, self-contained, numbered units. Articles longer than `max_chars` are split on their own numbered sub-clauses (`1.`, `2.`, `1-1.`, ...) rather than a blind character cut, respecting the law's own internal structure.

Both are run here so we can report chunk counts for each, as required.

In [4]:
all_articles = labor_articles + tax_articles

rc_400 = lib.recursive_char_chunks(all_articles, chunk_size=400, chunk_overlap=50)
rc_800 = lib.recursive_char_chunks(all_articles, chunk_size=800, chunk_overlap=100)
rc_1600 = lib.recursive_char_chunks(all_articles, chunk_size=1600, chunk_overlap=150)
article_based = lib.article_based_chunks(all_articles, max_chars=1600)

print(f"{len(all_articles)} source articles\n")
print(f"recursive_char (size=400):  {len(rc_400)} chunks")
print(f"recursive_char (size=800):  {len(rc_800)} chunks")
print(f"recursive_char (size=1600): {len(rc_1600)} chunks")
print(f"article_based (max=1600):   {len(article_based)} chunks")

626 source articles

recursive_char (size=400):  6361 chunks
recursive_char (size=800):  3080 chunks
recursive_char (size=1600): 1577 chunks
article_based (max=1600):   1298 chunks


In [5]:
import json

print("Sample article_based chunk (metadata):")
print(json.dumps(article_based[50].__dict__, ensure_ascii=False, indent=2)[:800])

Sample article_based chunk (metadata):
{
  "chunk_id": "labor_code:32:art:0",
  "source_file": "labor_code_raw.html",
  "law": "labor_code",
  "article_number": "32",
  "section_title": "Глава 4. ТРУДОВОЙ ДОГОВОР / Статья 32. Документы, необходимые для заключения трудового договора",
  "document_version": "adilet.zan.kz snapshot 2026-07-08",
  "text": "Статья 32. Документы, необходимые для заключения трудового договора 1. Для заключения трудового договора необходимы следующие документы: 1) удостоверение личности гражданина Республики Казахстан или паспорт гражданина Республики Казахстан (свидетельство о рождении для лиц, не достигших шестнадцатилетнего возраста). Кандасы представляют удостоверение кандаса, выданное местными исполнительными органами; 2) вид на жительство иностранца в Республике Казахстан или удостоверение лица б


## 3. Persist chunks + metadata

`scripts/build_index.py` is the idempotent, reproducible entry point that regenerates `data/chunks.jsonl` from the raw HTML on every run (same inputs -> same output; embedding cache below it makes re-running cheap). This notebook cell writes the same production default (`article_based`, chosen because it scored the highest recall@5 in the ablation - see `notebooks/02_ablation.ipynb`) so the artifact on disk always matches what the API serves.

In [6]:
chunks_path = ROOT / "data/chunks.jsonl"
with open(chunks_path, "w", encoding="utf-8") as f:
    for c in article_based:
        f.write(json.dumps(c.__dict__, ensure_ascii=False) + "\n")
print(f"Persisted {len(article_based)} chunks to {chunks_path}")

Persisted 1298 chunks to C:\Users\1204n\Music\raid_w12\data\chunks.jsonl
